https://www.sgx.com/securities/etf-screener

# Setup

In [ ]:
import pandas as pd

pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 20)

from IPython.display import display, HTML
from IPython import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
from datetime import datetime
import xlwings as xw
import time
import glob
import re

# import warnings
# warnings.filterwarnings("ignore")
import hashlib

In [16]:
import os
import pandas as pd
import numpy as np

# Define the folder path
folder_path = os.path.expanduser("~/Downloads_win")
# Get all files that start with 'export' and end with '.xlsx'
export_files = [
    f for f in os.listdir(folder_path) if f.startswith("myData") and f.endswith(".csv")
]
# Check if any matching files exist
if not export_files:
    raise FileNotFoundError("No 'export*.xlsx' files found in the specified folder.")
# Find the latest file based on modification time
latest_file = max(
    export_files, key=lambda f: os.path.getmtime(os.path.join(folder_path, f))
)
# Build the full path to the latest file
latest_file_path = os.path.join(folder_path, latest_file)
# Read the Excel file using pandas
df = pd.read_csv(latest_file_path)

# Display the result
print(f"Loaded file: {latest_file_path}")

Loaded file: /home/mrsmmori/Downloads_win/myData.csv


In [17]:
df = df.replace("-", np.nan)

for col in df.columns:
    try:
        df[col] = df[col].str.replace(",", "")
        df[col] = pd.to_numeric(df[col])
    except Exception:
        pass  #

In [18]:
df["Geographical Focus"].value_counts()

Geographical Focus
China               29
Singapore           17
Asia                15
Asia Pacific         8
Vietnam              3
India                3
Gold                 2
USA                  2
Japan                2
Saudi Arabia         1
Indonesia            1
Emerging Markets     1
Name: count, dtype: int64

# df -> df_usd, df_sgd

In [19]:
df_usd = df[(df["CCY"] == "USD") | (df["CCY"] == "USD")].copy()
df_sgd = df[(df["CCY"] == "SGD") | (df["CCY"] == "SGD")].copy()

# df_sgd -> df

In [28]:
df = df_sgd.copy()
df = df[df["Sustainability linked"] == "No"]

In [29]:
df["Geographical Focus"].value_counts()

Geographical Focus
China           13
Singapore        9
Asia             6
Asia Pacific     2
Gold             1
Japan            1
Vietnam          1
Name: count, dtype: int64

In [30]:
df["Asset Class"].value_counts()

Asset Class
Equities        20
Fixed Income     8
REITs            4
Commodities      1
Name: count, dtype: int64

In [36]:
df["Income Treatment"].value_counts()

Income Treatment
Distributing    20
Accumulating    12
Name: count, dtype: int64

In [31]:
df.columns

Index(['Trading Name', 'Trading Code', 'CCY', 'Underlying Benchmark', 'Close', 'Last', 'Chng %', 'Val ($M)',
       'Yield (%)', 'TER (%)', 'Documents', 'Fund Manager', 'TR 1M (%)', 'TR 3M (%)', 'TR 1Y (%)', 'Ann. TR 1Y (%)',
       'Ann. TR 3Y (%)', 'Asset Class', 'Geographical Focus', 'Income Treatment', 'Sustainability linked',
       'Management Style', 'CPF Eligibility', 'SIP/EIP', 'Tick Size'],
      dtype='object')

In [34]:
df = df[
    [
        "Trading Name",
        "Trading Code",
        #'CCY',
        "Underlying Benchmark",
        #'Close',
        #'Last',
        #'Chng %',
        #'Val ($M)',
        "Yield (%)",
        #'TER (%)',
        #'Documents',
        #'Fund Manager',
        "TR 1M (%)",
        "TR 3M (%)",
        "TR 1Y (%)",
        "Ann. TR 1Y (%)",
        "Ann. TR 3Y (%)",
        "Asset Class",
        "Geographical Focus",
        "Income Treatment",
        #'Sustainability linked',
        "Management Style",
        #'CPF Eligibility',
        #'SIP/EIP',
        #'Tick Size',
    ]
]

In [42]:
grouped = df.groupby(["Asset Class", "Income Treatment"])

for (asset_class, income), group_df in grouped:
    print(f"=== Asset Class: {asset_class}, Income Treatment: {income} ===")
    group_df

=== Asset Class: Equities, Income Treatment: Accumulating ===


,Trading Name,Trading Code,Underlying Benchmark,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style
3,Lion-OCBC Sec HSTECH S$,HST,Hang Seng TECH Index,NaN,0.55,6.94,61.74,61.74,7.35,Equities,China,Accumulating,PASSIVE
13,CSOP Star&Chinext50 S$,SCY,CSI STAR and ChiNext 50 Index,NaN,NaN,NaN,NaN,NaN,NaN,Equities,China,Accumulating,PASSIVE
14,XT MSCHINA S$,TID,MSCI China TR Net Daily USD Index,NaN,1.08,10.76,45.53,45.53,5.82,Equities,China,Accumulating,PASSIVE
15,Amova Asia exJC S$,A93,MSCI AC Asia ex Japan ex China Index,NaN,NaN,NaN,NaN,NaN,NaN,Equities,Asia,Accumulating,PASSIVE
30,AmovaEFund ChiNext S$,CXT,ChiNext Index,NaN,NaN,NaN,NaN,NaN,NaN,Equities,China,Accumulating,PASSIVE
32,A Lion-Nomura Japan S$,JJJ,Topix Index,NaN,6.70,10.70,21.00,21.00,NaN,Equities,Japan,Accumulating,ACTIVE
33,CF VN 30 SC ETF SG$,VNM,iEdge Vietnam 30 Sector Cap USD Index (NTR),NaN,14.86,31.68,39.34,39.34,NaN,Equities,Vietnam,Accumulating,PASSIVE
35,UOBAM PINGAN CHINEXT S$,CXS,ChiNext Index,NaN,19.97,39.96,74.23,74.23,NaN,Equities,China,Accumulating,PASSIVE
44,CSOP SEA TECH ETF S$,SQQ,iEdge Southeast Asia+ TECH Index,NaN,7.16,12.44,12.56,12.56,NaN,Equities,Asia,Accumulating,PASSIVE
69,Lion-CM EM Asia S$,EAA,iEdge Emerging Asia Select 50 Index,NaN,NaN,NaN,NaN,NaN,NaN,Equities,Asia,Accumulating,PASSIVE


=== Asset Class: Equities, Income Treatment: Distributing ===


,Trading Name,Trading Code,Underlying Benchmark,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style
0,STI ETF,ES3,Straits Times Index,4.17,2.49,10.48,30.48,30.48,14.80,Equities,Singapore,Distributing,PASSIVE
9,Amova STI ETF,G3B,Straits Times Index,4.09,2.50,10.67,31.11,31.11,14.81,Equities,Singapore,Distributing,PASSIVE
10,Lion-OSPL China L S$,YYY,Hang Seng Stock Connect China 80 Index,3.05,1.52,7.98,32.61,32.61,5.94,Equities,China,Distributing,PASSIVE
17,Lion-OSPL APAC Fin S$,YLD,iEdge APAC Financials Dividend Plus Index,6.44,4.43,5.85,17.94,17.94,NaN,Equities,Asia Pacific,Distributing,PASSIVE
22,UOBAM FTSE CN A50 S$,JK8,FTSE China A50 Index,NaN,NaN,NaN,NaN,NaN,NaN,Equities,China,Distributing,PASSIVE
34,Lion-CM CSI Div S$,INC,CSI Dividend Index,NaN,-99.89,-99.89,NaN,NaN,NaN,Equities,China,Distributing,PASSIVE
39,PHIL SING INC,OVQ,Morningstar Singapore Yield Focus Index,3.39,2.11,7.78,26.43,26.43,13.30,Equities,Singapore,Distributing,PASSIVE
59,CGS FG CSI1000 S$,GRO,CSI 1000 Index,NaN,15.17,25.95,54.98,54.98,NaN,Equities,China,Distributing,PASSIVE
61,CSOP DIV ETF S$,SHD,SSE Dividend Index,NaN,0.38,2.63,7.55,7.55,NaN,Equities,China,Distributing,PASSIVE


=== Asset Class: Fixed Income, Income Treatment: Accumulating ===


,Trading Name,Trading Code,Underlying Benchmark,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style
37,XT SingGovBond SG$,KV4,FTSE Singapore Government Bond Index,NaN,2.64,5.78,10.25,10.25,5.86,Fixed Income,Singapore,Accumulating,PASSIVE


=== Asset Class: Fixed Income, Income Treatment: Distributing ===


,Trading Name,Trading Code,Underlying Benchmark,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style
7,ABF SG BOND ETF,A35,iBoxx ABF Singapore Bond Index,2.24,2.37,5.49,9.92,9.92,6.15,Fixed Income,Singapore,Distributing,PASSIVE
8,Amova SGD IGBond ETF,MBH,iBoxx SGD Non-Sovereigns Large Cap Investment ...,3.22,1.14,3.98,8.50,8.50,6.20,Fixed Income,Singapore,Distributing,PASSIVE
12,IS ASIA HYG S$D,QL3,Bloomberg Asia USD High Yield Diversified Cred...,7.23,NaN,NaN,NaN,NaN,NaN,Fixed Income,Asia,Distributing,PASSIVE
21,PHILLIP MM S$,MMS,FTSE SGD 3-month SOR Index,2.88,0.15,0.56,1.78,1.78,2.65,Fixed Income,Singapore,Distributing,PASSIVE
29,IS ASIA BND S$D,QL2,J.P. Morgan Asia Credit Index - Core,4.24,NaN,NaN,NaN,NaN,NaN,Fixed Income,Asia,Distributing,PASSIVE
42,ICBC CSOP CGB ETF S$,CYC,FTSE Chinese Government Bond Index,2.95,0.37,0.50,1.54,1.54,0.01,Fixed Income,China,Distributing,PASSIVE
45,Amova-ICBCSG CNB S$,ZHS,ChinaBond ICBC 1-10 Year Treasury and Policy B...,2.37,0.31,0.05,0.57,0.57,-0.69,Fixed Income,China,Distributing,PASSIVE


=== Asset Class: REITs, Income Treatment: Distributing ===


,Trading Name,Trading Code,Underlying Benchmark,Yield (%),TR 1M (%),TR 3M (%),TR 1Y (%),Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Management Style
4,LION-PHILLIP S-REIT,CLR,Morningstar Singapore REIT Yield Focus Index SM,5.67,0.36,8.63,-0.23,-0.23,-1.81,REITs,Singapore,Distributing,PASSIVE
5,Amova-STC Asia REIT,CFA,FTSE EPRA/NAREIT Asia ex Japan Net Total Retur...,5.45,1.55,9.09,6.53,6.53,-1.40,REITs,Asia,Distributing,PASSIVE
11,CSOP iEdge SREIT ETF S$,SRT,iEdge SREIT Leaders Index,5.73,3.85,11.22,2.55,2.55,-1.13,REITs,Singapore,Distributing,PASSIVE
23,PHIL AP DIV REIT S$D,BYJ,iEdge APAC ex Japan Dividend Leaders REIT Index,4.28,NaN,NaN,NaN,NaN,NaN,REITs,Asia Pacific,Distributing,PASSIVE


In [ ]:
code = "VND"
url = f"https://www.example.com/stocks/{code}"
print(url)

# df_usd -> df

In [43]:
df = df_usd.copy()
df = df[df["Sustainability linked"] == "No"]

In [44]:
grouped = df.groupby(["Asset Class", "Income Treatment"])

for (asset_class, income), group_df in grouped:
    print(f"=== Asset Class: {asset_class}, Income Treatment: {income} ===")
    group_df

=== Asset Class: Equities, Income Treatment: Accumulating ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
28,XT MSCHINA US$,LG9,USD,MSCI China TR Net Daily USD Index,21.00,21.00,1.79,0.03,NaN,0.65,...,47.90,8.73,Equities,China,Accumulating,No,PASSIVE,No,EIP,0.01
31,CF VN 30 SC ETF US$,VND,USD,iEdge Vietnam 30 Sector Cap USD Index (NTR),1.28,1.26,2.52,0.03,NaN,2.39,...,50.79,NaN,Equities,Vietnam,Accumulating,No,PASSIVE,No,EIP,0.00
38,Amundi MSIndia Sw US$,G1N,USD,MSCI India Net Total Return Index,32.10,32.23,0.41,0.00,NaN,0.85,...,-11.87,7.62,Equities,India,Accumulating,No,PASSIVE,No,SIP,0.01
41,XT MSINDO US$,KJ7,USD,MSCI Daily TRN Net Emerging Markets Indonesia ...,13.47,13.18,1.39,0.00,NaN,0.65,...,-16.77,-4.46,Equities,Indonesia,Accumulating,No,PASSIVE,No,SIP,0.01
46,XT Vietnam US$,HD9,USD,FTSE Vietnam Index,37.67,37.61,1.65,0.00,NaN,0.85,...,43.57,2.79,Equities,Vietnam,Accumulating,No,PASSIVE,No,SIP,0.01
49,A Lion-Nomura Japan US$,JUS,USD,Topix Index,0.97,0.97,0.00,0.00,NaN,0.98,...,30.13,NaN,Equities,Japan,Accumulating,No,ACTIVE,No,EIP,0.00
50,Amova Asia exJC US$,A94,USD,MSCI AC Asia ex Japan ex China Index,0.89,0.86,0.00,0.00,NaN,NaN,...,NaN,NaN,Equities,Asia,Accumulating,No,PASSIVE,No,EIP,0.00
57,AmovaEFund ChiNext US$,CXO,USD,ChiNext Index,0.19,0.00,0.00,0.00,NaN,NaN,...,NaN,NaN,Equities,China,Accumulating,No,PASSIVE,No,EIP,0.00
58,Amundi EM Mkt Sw US$,H1N,USD,MSCI Emerging Market Net Total Return Index,16.67,16.25,0.00,0.00,NaN,0.55,...,17.18,10.54,Equities,Emerging Markets,Accumulating,No,PASSIVE,No,SIP,0.01
63,CSOP SEA TECH ETF US$,SQU,USD,iEdge Southeast Asia+ TECH Index,0.96,0.93,0.00,0.00,NaN,1.41,...,13.89,NaN,Equities,Asia,Accumulating,No,PASSIVE,No,EIP,0.00


=== Asset Class: Equities, Income Treatment: Distributing ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
20,SPDR S&P500 US$,S27,USD,S&P 500 Index,658.05,658.04,0.62,0.10,1.20,0.00,...,16.95,18.67,Equities,USA,Distributing,No,PASSIVE,No,SIP,0.01
40,Lion-OSPL APAC Fin US$,YLU,USD,iEdge APAC Financials Dividend Plus Index,0.99,0.99,3.22,0.00,6.44,1.08,...,44.26,NaN,Equities,Asia Pacific,Distributing,No,PASSIVE,No,EIP,0.00
60,CGS FG CSI1000 US$,GRU,USD,CSI 1000 Index,1.22,1.22,0.00,0.00,NaN,1.96,...,45.19,NaN,Equities,China,Distributing,No,PASSIVE,No,EIP,0.00
79,SPDR DJIA US$,D07,USD,Dow Jones Industrial AverageSM (DJIASM) Index,462.13,459.00,0.00,0.00,1.49,0.16,...,11.95,14.43,Equities,USA,Distributing,No,PASSIVE,No,SIP,0.01
81,UOBAM FTSE CN A50 US$,VK8,USD,FTSE China A50 Index,1.89,1.64,0.00,0.00,NaN,0.84,...,NaN,NaN,Equities,China,Distributing,No,PASSIVE,No,EIP,0.00


=== Asset Class: Fixed Income, Income Treatment: Accumulating ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
36,SPDR JPM SaudiAggBd US$,KSB,USD,J.P. Morgan Saudi Arabia Aggregate Index,31.46,31.39,0.93,0.01,NaN,0.37,...,NaN,NaN,Fixed Income,Saudi Arabia,Accumulating,No,PASSIVE,No,EIP,0.00
53,Amova-ICBCSG CNB US$,ZHD,USD,ChinaBond ICBC 1-10 Year Treasury and Policy B...,0.84,0.83,0.00,0.00,NaN,0.30,...,NaN,NaN,Fixed Income,China,Accumulating,No,PASSIVE,No,EIP,0.00
64,ICBC CSOP CGB ETF US$A,CYX,USD,FTSE Chinese Government Bond Index,11.02,10.82,0.00,0.00,NaN,0.26,...,2.80,NaN,Fixed Income,China,Accumulating,No,PASSIVE,No,EIP,0.01


=== Asset Class: Fixed Income, Income Treatment: Distributing ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
1,IS ASIA HYG US$,O9P,USD,Bloomberg Asia USD High Yield Diversified Cred...,6.70,6.70,0.30,4.54,7.23,0.50,...,7.16,8.99,Fixed Income,Asia,Distributing,No,PASSIVE,No,EIP,0.01
65,ICBC CSOP CGB ETF US$D,CYB,USD,FTSE Chinese Government Bond Index,10.06,10.05,0.00,0.00,2.95,0.26,...,3.29,3.07,Fixed Income,China,Distributing,No,PASSIVE,No,EIP,0.01
66,IS ASIA BND US$,N6M,USD,J.P. Morgan Asia Credit Index - Core,9.91,9.90,0.00,0.00,4.24,0.20,...,3.65,5.80,Fixed Income,Asia,Distributing,No,PASSIVE,No,EIP,0.01
78,PHILLIP MM US$D,MMT,USD,FTSE SGD 3-month SOR Index,80.61,79.76,0.00,0.00,2.88,0.28,...,NaN,NaN,Fixed Income,Singapore,Distributing,No,PASSIVE,No,EIP,0.00


=== Asset Class: REITs, Income Treatment: Distributing ===


,Trading Name,Trading Code,CCY,Underlying Benchmark,Close,Last,Chng %,Val ($M),Yield (%),TER (%),...,Ann. TR 1Y (%),Ann. TR 3Y (%),Asset Class,Geographical Focus,Income Treatment,Sustainability linked,Management Style,CPF Eligibility,SIP/EIP,Tick Size
43,CSOP iEdge SREIT ETF US$,SRU,USD,iEdge SREIT Leaders Index,0.61,0.61,0.66,0.00,5.73,0.60,...,3.78,2.29,REITs,Singapore,Distributing,No,PASSIVE,No,EIP,0.00
54,Amova-STC A_REIT US$,COI,USD,FTSE EPRA/NAREIT Asia ex Japan Net Total Retur...,0.66,0.65,0.00,0.00,5.45,0.55,...,NaN,NaN,REITs,Asia,Distributing,No,PASSIVE,Yes,EIP,0.00
74,PHIL AP DIV REIT US$,BYI,USD,iEdge APAC ex Japan Dividend Leaders REIT Index,0.94,0.93,0.00,0.00,4.28,1.54,...,6.96,1.65,REITs,Asia Pacific,Distributing,No,PASSIVE,No,EIP,0.00
